In [4]:
import pandas as pd 
import numpy as np 

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hrishitpatil/flight-data-2024")

print("Path to dataset files:", path)

Path to dataset files: /Users/megantimmes/.cache/kagglehub/datasets/hrishitpatil/flight-data-2024/versions/2


In [6]:
import os

csv_path = os.path.join(path, "flight_data_2024.csv")
parquet_path = "data/flight_data_2024.parquet"

dtypes = {
    # low-cardinality text -> category (big memory savings)
    "op_unique_carrier": "category", "origin": "category", "dest": "category",
    "origin_city_name": "category", "dest_city_name": "category",
    "origin_state_nm": "category", "dest_state_nm": "category",
    "cancellation_code": "category",
    # small integers
    "year": "int16", "month": "int8", "day_of_month": "int8", "day_of_week": "int8",
    "crs_dep_time": "int16", "crs_arr_time": "int16",
    "cancelled": "int8", "diverted": "int8",
    "carrier_delay": "int32", "weather_delay": "int32", "nas_delay": "int32",
    "security_delay": "int32", "late_aircraft_delay": "int32",
    # floats with NaNs -> float32
    **{c: "float32" for c in [
        "op_carrier_fl_num", "dep_time", "dep_delay", "taxi_out", "wheels_off",
        "wheels_on", "taxi_in", "arr_time", "arr_delay", "crs_elapsed_time",
        "actual_elapsed_time", "air_time", "distance"]},
}

# Read category columns as plain strings first, then convert after loading.
# (pandas reads in chunks; mostly-empty columns like cancellation_code get
# mismatched category types across chunks otherwise.)
cat_cols = [k for k, v in dtypes.items() if v == "category"]
read_dtypes = {k: ("str" if v == "category" else v) for k, v in dtypes.items()}

if os.path.exists(parquet_path):
    df = pd.read_parquet(parquet_path)          # fast path (seconds)
else:
    df = pd.read_csv(csv_path, dtype=read_dtypes, parse_dates=["fl_date"])
    df[cat_cols] = df[cat_cols].astype("category")
    os.makedirs("data", exist_ok=True)
    df.to_parquet(parquet_path, index=False)    # one-time conversion

print(df.shape)
df.info(memory_usage="deep")

(7079081, 35)
<class 'pandas.DataFrame'>
RangeIndex: 7079081 entries, 0 to 7079080
Data columns (total 35 columns):
 #   Column               Dtype         
---  ------               -----         
 0   year                 int16         
 1   month                int8          
 2   day_of_month         int8          
 3   day_of_week          int8          
 4   fl_date              datetime64[us]
 5   op_unique_carrier    category      
 6   op_carrier_fl_num    float32       
 7   origin               category      
 8   origin_city_name     category      
 9   origin_state_nm      category      
 10  dest                 category      
 11  dest_city_name       category      
 12  dest_state_nm        category      
 13  crs_dep_time         int16         
 14  dep_time             float32       
 15  dep_delay            float32       
 16  taxi_out             float32       
 17  wheels_off           float32       
 18  wheels_on            float32       
 19  taxi_in           

In [7]:
df.head()

,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,...,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024,1,1,1,2024-01-01,9E,4814.0,JFK,"New York, NY",New York,...,0,136.0,122.0,84.0,509.0,0,0,0,0,0
1,2024,1,1,1,2024-01-01,9E,4815.0,MSP,"Minneapolis, MN",Minnesota,...,0,130.0,114.0,88.0,622.0,0,0,0,0,0
2,2024,1,1,1,2024-01-01,9E,4817.0,JFK,"New York, NY",New York,...,0,106.0,90.0,61.0,288.0,0,0,0,0,0
3,2024,1,1,1,2024-01-01,9E,4817.0,RIC,"Richmond, VA",Virginia,...,0,111.0,76.0,51.0,288.0,0,0,0,0,0
4,2024,1,1,1,2024-01-01,9E,4818.0,DTW,"Detroit, MI",Michigan,...,0,79.0,70.0,45.0,237.0,0,0,0,0,0
